<div dir="rtl">

# NB03-M01 — Quran Canonical Ingestion and Acceptance

## هدف
تبدیل فایل Raw پذیرفته‌شده از نظر ساختاری به JSONL آیه‌محور، بدون Normalization متن عربی و با Lineage مستقیم به Raw SHA-256.

</div>

In [1]:
# NB03-C01 — Imports and project root
import json
import sys
from pathlib import Path

current = Path.cwd().resolve()
project_root = next(
    (p for p in [current, *current.parents] if (p / ".git").exists() and (p / "data").exists()),
    None,
)
if project_root is None:
    raise RuntimeError("QRAG project root not found.")

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from qrag_phase1.foundation import collect_git_info, write_json_atomic
from qrag_phase1.quran import (
    build_canonical_quran_records,
    parse_quran_pipe_file,
    read_jsonl,
    sha256_file,
    validate_quran_records,
    write_jsonl_atomic,
)

assert collect_git_info(project_root)["branch"] == "qrag-phase1"

<div dir="rtl">

# NB03-M02 — Upstream Gate

NB02 باید روی همین Raw SHA-256 PASS کرده باشد.

</div>

In [2]:
# NB03-C02 — Load upstream acceptance
reports_dir = project_root / "reports/phase1"
upstream_path = reports_dir / "quran_canonical_inventory_acceptance_v1.json"
contract_path = project_root / "config/phase1/quran_canonical_contract_v1.json"
raw_path = project_root / "data/raw/quran_arabic_uthmani.txt"

assert upstream_path.is_file(), "Run NB02 first."
upstream = json.loads(upstream_path.read_text(encoding="utf-8"))
contract = json.loads(contract_path.read_text(encoding="utf-8"))

assert upstream["acceptance_status"] in {"PASS", "PASS_WITH_LIMITATIONS"}
assert upstream["raw_sha256"] == sha256_file(raw_path)

<div dir="rtl">

# NB03-M03 — Deterministic Canonical Build

هر Record شناسه quran:surah:ayah و Raw hash دارد.

</div>

In [3]:
# NB03-C03 — Build canonical records
parsed = parse_quran_pipe_file(raw_path)
validation = validate_quran_records(
    parsed["records"],
    contract["expected_verse_counts_by_surah"],
)
assert validation["status"] == "PASS"

raw_sha256 = sha256_file(raw_path)
canonical_rows = build_canonical_quran_records(
    parsed["records"],
    contract["source_id"],
    raw_sha256,
)

assert len(canonical_rows) == 6236
assert canonical_rows[0]["verse_id"] == "quran:1:1"
assert canonical_rows[-1]["verse_id"] == "quran:114:6"

<div dir="rtl">

# NB03-M04 — Persist and Reload

خروجی در data/phase1/processed ساخته و دوباره از Disk خوانده می‌شود.

</div>

In [4]:
# NB03-C04 — Atomic write and independent reload
output_path = project_root / contract["canonical_output"]["path"]
write_jsonl_atomic(output_path, canonical_rows)

reloaded = read_jsonl(output_path)
assert len(reloaded) == len(canonical_rows)

for index in (0, 1, 100, 1000, 6235):
    assert reloaded[index] == canonical_rows[index]

output_sha256 = sha256_file(output_path)
print("Canonical output:", output_path.relative_to(project_root))
print("SHA-256:", output_sha256)

Canonical output: data/phase1/processed/quran_canonical_v1.jsonl
SHA-256: 12b25d029e071b3d1ee700b8176159a63498b0037777e53fe0031fddb658b973


<div dir="rtl">

# NB03-M05 — Verbatim Text Fidelity

Canonical field باید دقیقاً برابر text خوانده‌شده از Raw باشد.

</div>

In [5]:
# NB03-C05 — Verbatim text fidelity
text_mismatches = []

for source, canonical in zip(parsed["records"], reloaded):
    if source["text"] != canonical["arabic_uthmani"]:
        text_mismatches.append(canonical["verse_id"])
        if len(text_mismatches) >= 20:
            break

assert text_mismatches == []
print("Verbatim text fidelity: PASS")

Verbatim text fidelity: PASS


<div dir="rtl">

# NB03-M06 — Canonical Manifest and Acceptance

این Acceptance فقط متن عربی Canonical فاز یک را فعال می‌کند؛ ترجمه هنوز در NB04/NB05 جداگانه بررسی می‌شود.

</div>

In [6]:
# NB03-C06 — Manifest and acceptance
manifest = {
    "schema_version": "qrag_quran_canonical_manifest_v1",
    "release_id": "qrag_phase1_quran_canonical_v1",
    "source_id": contract["source_id"],
    "raw_path": str(raw_path.relative_to(project_root)),
    "raw_sha256": raw_sha256,
    "canonical_path": str(output_path.relative_to(project_root)),
    "canonical_sha256": output_sha256,
    "record_count": len(reloaded),
    "first_verse_id": reloaded[0]["verse_id"],
    "last_verse_id": reloaded[-1]["verse_id"],
    "text_transformation": "NONE_VERBATIM_COPY_TO_STRUCTURED_RECORD",
    "status": "ACCEPTED_FOR_QRAG_PHASE1_RESEARCH",
    "known_limitation": (
        "Historical remote-download checksum was not preserved in the "
        "legacy ingestion; local raw SHA-256 is frozen here."
    ),
}

manifest_path = project_root / "artifacts/phase1/quran/quran_canonical_manifest_v1.json"
write_json_atomic(manifest_path, manifest)

acceptance = {
    "schema_version": "qrag_nb03_acceptance_v1",
    "objective_status": "ACHIEVED",
    "acceptance_status": "PASS_WITH_LIMITATIONS",
    "failed_checks": [],
    "record_count": 6236,
    "canonical_sha256": output_sha256,
    "active_quran_source_id": contract["source_id"],
    "next_notebook": "notebooks/phase1/04_translation_inventory.ipynb",
}
acceptance_path = reports_dir / "quran_canonical_ingestion_acceptance_v1.json"
write_json_atomic(acceptance_path, acceptance)

assert json.loads(manifest_path.read_text(encoding="utf-8")) == manifest
assert json.loads(acceptance_path.read_text(encoding="utf-8")) == acceptance
print(json.dumps(acceptance, ensure_ascii=False, indent=2))

{
  "schema_version": "qrag_nb03_acceptance_v1",
  "objective_status": "ACHIEVED",
  "acceptance_status": "PASS_WITH_LIMITATIONS",
  "failed_checks": [],
  "record_count": 6236,
  "canonical_sha256": "12b25d029e071b3d1ee700b8176159a63498b0037777e53fe0031fddb658b973",
  "active_quran_source_id": "quran:uthmani:legacy-repo-v1",
  "next_notebook": "notebooks/phase1/04_translation_inventory.ipynb"
}


<div dir="rtl">

# NB03-M07 — جمع‌بندی

پس از Run All موفق، قرآن عربی Phase 1 یک Artifact مستقل، Hash‌شده و قابل ردیابی خواهد داشت. هر نوع Normalization در NB11 و روی فیلد مشتق‌شده انجام می‌شود، نه روی این Canonical text.

</div>